# Frosty Friday Week 99 — AI_EXTRACT & AI_PARSE_DOCUMENT

## Challenge: Document Intelligence with Snowflake Cortex AI

あなたは製造業の品質管理部門に所属するデータエンジニアです。
社内の各部門から届く様々な形式のPDF文書（請求書、品質検査報告書、納品書、議事録）を手動で確認・転記する作業に多くの時間を費やしています。

Snowflake Cortex AI の `AI_EXTRACT` と `AI_PARSE_DOCUMENT` を使って、この作業を自動化してください。

### Requirements

1. **ステージ作成 & ファイルアップロード** — `FFDB.PUBLIC.FF_STAGE` に4つのPDFをアップロード
2. **フィールド抽出（AI_EXTRACT 簡易形式）** — 請求書/品質検査報告/納品書から主要フィールドを抽出
3. **テーブル抽出（AI_EXTRACT schema形式）** — 明細テーブル/検査結果テーブル/決定事項を配列で抽出
4. **推論抽出（従来OCRでは不可能）** — 判断・計算・異フォーマット同一質問
5. **AI_COMPLETE連携** — AI_PARSE_DOCUMENT → AI_COMPLETE で議事録を3行に要約

### 提供ファイル
- `invoice_sample.pdf` — 日本語請求書（明細テーブル付き）
- `quality_inspection.pdf` — 品質検査結果報告書（寸法検査・性能試験テーブル付き）
- `delivery_note.pdf` — 納品書
- `meeting_minutes.pdf` — 議事録（決定事項テーブル付き）

---
## 解答

## 0. セットアップ

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()
cur = session.connection.cursor()
print("Session connected")

In [ ]:
cur.execute("USE WAREHOUSE COMPUTE_WH")
cur.execute("CREATE DATABASE IF NOT EXISTS FFDB")
cur.execute("USE DATABASE FFDB")
cur.execute("USE SCHEMA PUBLIC")
cur.execute("CREATE STAGE IF NOT EXISTS FF_STAGE DIRECTORY = (ENABLE = TRUE) ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')")
print("ステージ FF_STAGE を作成しました")

### 📂 PDFファイルのアップロード（GUI操作）

以下の手順で `pdf_samples` フォルダ内のファイルをステージにアップロードしてください。

1. [Snowsight](https://app.snowflake.com) にログイン
2. 左メニューから **Data** → **Databases** → **FFDB** → **PUBLIC** → **Stages** → **FF_STAGE** を開く
3. 右上の **+ Files** ボタンをクリック
4. `pdf_samples` フォルダ内のPDFファイルをすべて選択してアップロード
5. アップロード完了を確認したら、次のセルを実行してファイル一覧を確認します

In [ ]:
cur.execute("LIST @FF_STAGE")
files_df = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
print(f"ステージ内ファイル数: {len(files_df)}")
display(files_df[['name', 'size']] if len(files_df) > 0 else "ファイルなし")

---
## ウォームアップ: テキスト直接入力で AI_EXTRACT
> ステージ不要！テキストを渡すだけでJSON抽出

In [ ]:
sql = """
SELECT AI_EXTRACT(
    text => '山田太郎は東京都渋谷区神宮前1-2-3に住んでおり、株式会社スノーフレークでデータエンジニアとして勤務しています。メールアドレスはtaro.yamada@snowflake.com、電話番号は03-1234-5678です。入社日は2023年4月1日で、年収は850万円です。',
    responseFormat => ['person_name', 'address', 'company', 'job_title', 'email', 'phone', 'start_date', 'annual_salary']
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(pd.DataFrame([result]))

---
## Part 1: AI_EXTRACT - PDFからフィールド抽出
> `TO_FILE('@ステージ', 'ファイル名')` でPDFを指定し、`responseFormat`に抽出したいフィールドと質問文を記述

**ポイント:**
- responseFormat の Value に「質問文」を書くと精度UP
- 「請求先ではなく請求元」のような否定指示が有効
- 「YYYY-MM-DD形式で」等のフォーマット指定が可能

### 1a. 請求書

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'invoice_sample.pdf'),
    responseFormat => {
        'company_name': '請求元の会社名は？（請求先ではなく請求元）',
        'invoice_number': '請求書番号は？',
        'invoice_date': '請求日は？YYYY-MM-DD形式で',
        'total_amount': '税込の合計金額は？数値のみ（円マーク・カンマなし）',
        'payment_due_date': '支払期日は？YYYY-MM-DD形式で',
        'bank_account': '振込先の銀行口座情報は？'
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(pd.DataFrame([result]))

### 1b. 品質検査報告書（製造業ユースケース）

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'quality_inspection.pdf'),
    responseFormat => {
        'product_name': '検査対象の製品名は？',
        'lot_number': 'ロット番号は？',
        'inspection_date': '検査日は？YYYY-MM-DD形式で',
        'inspector': '検査員の名前は？',
        'overall_result': '総合判定の結果は？（合格 or 不合格）',
        'defect_rate': '不良率は？',
        'cpk_value': 'Cpk値は？',
        'manufacturing_line': '製造ラインは？'
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(pd.DataFrame([result]))

### 1c. 納品書

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'delivery_note.pdf'),
    responseFormat => {
        'delivery_number': '納品書番号は？',
        'delivery_date': '納品日は？YYYY-MM-DD形式で',
        'customer_name': '納品先の会社名は？',
        'supplier_name': '納品元の会社名は？',
        'po_number': '発注番号は？',
        'acceptance_deadline': '検収期限は？YYYY-MM-DD形式で'
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(pd.DataFrame([result]))

---
## Part 2: AI_EXTRACT - 明細テーブル抽出 (schema形式)
> **schema形式とは？** `responseFormat`にJSON Schema構造で`type: 'array'`を指定することで、
> テーブル（複数行×複数列）のデータを配列として抽出できる書き方。
> Part 1の簡易形式（キー:質問ペア）は単一値向き、schema形式は繰り返しデータ向き。

**ポイント:**
- `column_ordering` でカラムの並び順を指定
- 各プロパティの `type: 'array'` が配列抽出のキー
- Pythonで `pd.DataFrame()` に渡せばそのままテーブル表示

### 2a. 請求書の明細テーブル

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'invoice_sample.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'invoice_number': { 'type': 'string', 'description': '請求書番号' },
                'total': { 'type': 'string', 'description': '合計金額（税込）' },
                'line_items': {
                    'type': 'object',
                    'description': '明細行のテーブル',
                    'column_ordering': ['item_name', 'quantity', 'unit_price', 'amount'],
                    'properties': {
                        'item_name': { 'description': '品名', 'type': 'array' },
                        'quantity': { 'description': '数量', 'type': 'array' },
                        'unit_price': { 'description': '単価', 'type': 'array' },
                        'amount': { 'description': '金額', 'type': 'array' }
                    }
                }
            }
        }
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(Markdown(f"**請求書番号:** {result.get('invoice_number')} / **合計:** {result.get('total')}"))
line_items = result.get('line_items', [])
display(pd.DataFrame(line_items) if isinstance(line_items, list) else pd.DataFrame([line_items]))

### 2b. 品質検査 - 寸法検査 & 性能試験テーブル

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'quality_inspection.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'dimension_inspection': {
                    'type': 'object',
                    'description': '寸法検査結果テーブル',
                    'column_ordering': ['item', 'spec', 'measured_avg', 'min_val', 'max_val', 'result'],
                    'properties': {
                        'item': { 'description': '検査項目', 'type': 'array' },
                        'spec': { 'description': '規格値', 'type': 'array' },
                        'measured_avg': { 'description': '実測平均', 'type': 'array' },
                        'min_val': { 'description': '最小値', 'type': 'array' },
                        'max_val': { 'description': '最大値', 'type': 'array' },
                        'result': { 'description': '判定', 'type': 'array' }
                    }
                },
                'performance_test': {
                    'type': 'object',
                    'description': '性能試験結果テーブル',
                    'column_ordering': ['test_item', 'criteria', 'measured', 'result'],
                    'properties': {
                        'test_item': { 'description': '試験項目', 'type': 'array' },
                        'criteria': { 'description': '基準値', 'type': 'array' },
                        'measured': { 'description': '測定値', 'type': 'array' },
                        'result': { 'description': '判定', 'type': 'array' }
                    }
                }
            }
        }
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(Markdown("**寸法検査結果:**"))
display(pd.DataFrame(result.get('dimension_inspection', [])))
display(Markdown("**性能試験結果:**"))
display(pd.DataFrame(result.get('performance_test', [])))

### 2c. 議事録 - 決定事項アクションアイテム

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'meeting_minutes.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'meeting_name': { 'type': 'string', 'description': '会議名' },
                'date': { 'type': 'string', 'description': '会議日時' },
                'attendees': { 'type': 'array', 'description': '出席者リスト' },
                'action_items': {
                    'type': 'object',
                    'description': '決定事項テーブル',
                    'column_ordering': ['action', 'owner', 'deadline'],
                    'properties': {
                        'action': { 'description': '決定事項', 'type': 'array' },
                        'owner': { 'description': '担当者', 'type': 'array' },
                        'deadline': { 'description': '期限', 'type': 'array' }
                    }
                }
            }
        }
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
attendees = result.get('attendees', [])
display(Markdown(f"**会議名:** {result.get('meeting_name')} / **日時:** {result.get('date')}"))
display(Markdown(f"**出席者:** {', '.join(attendees) if isinstance(attendees, list) else attendees}"))
display(pd.DataFrame(result.get('action_items', [])))

---
## Part 3: 推論抽出 — 従来AI OCRとの決定的な違い

| 観点 | 従来AI OCR | AI_EXTRACT |
|------|-----------|------------|
| 事前準備 | テンプレート定義（座標/アンカー） | 質問文を書くだけ |
| フォーマット変更 | テンプレート作り直し | 同じ質問で対応 |
| 推論・判断 | 不可能（見たまま抽出のみ） | 可能（文脈から推論） |
| 複数フォーマット | 個別テンプレート | 同一質問で対応 |
| 導入期間 | 数週間〜数ヶ月 | 数分（SQL 1行） |

### 3a. 品質検査報告から「判断」を抽出（文書に直接書かれていない情報）

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'quality_inspection.pdf'),
    responseFormat => {
        'is_shippable': 'この製品は出荷可能か？（合格かつ不良率0%なら「はい」）',
        'risk_items': '規格値に対して余裕が少ない（基準の80%以上を使っている）検査項目はあるか？あれば項目名を列挙',
        'improvement_vs_previous': '前回ロットと比較して改善した点は？'
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(Markdown("**推論抽出結果（文書に直接書かれていない情報をAIが判断）:**"))
display(pd.DataFrame([result]))

### 3b. 請求書から「計算・集約」を抽出（OCRでは不可能）

In [ ]:
sql = """
SELECT AI_EXTRACT(
    file => TO_FILE('@FF_STAGE', 'invoice_sample.pdf'),
    responseFormat => {
        'most_expensive_item': '最も高額な明細行の品名は？',
        'item_count': '明細行は全部で何行ある？数値のみ',
        'average_unit_price': '明細行の平均単価はいくらか？数値のみ'
    }
):response AS result
"""
cur.execute(sql)
result = json.loads(cur.fetchone()[0])
display(Markdown("**計算・集約抽出（従来OCRでは不可能）:**"))
display(pd.DataFrame([result]))

### 3c. テンプレート不要: 異なるフォーマットの文書に同じ質問
> `DIRECTORY(@ステージ名)` でステージ内の全ファイルを一覧取得し、
> `WHERE relative_path ILIKE '%.pdf'` でフィルタ → 各ファイルに同じAI_EXTRACTを適用。
> これにより **テンプレートなしで異なるフォーマットの文書を一括処理** できる。

In [ ]:
sql = """
SELECT
    relative_path AS file_name,
    AI_EXTRACT(
        file => TO_FILE('@FF_STAGE', relative_path),
        responseFormat => {
            'responsible_person': '文書の責任者または承認者は誰？',
            'deadline': '期限や支払期日など、最も重要な期日は？YYYY-MM-DD形式で',
            'risk_or_issue': 'この文書に記載されているリスクや課題は？なければ「なし」'
        }
    ):response AS extracted
FROM DIRECTORY(@FF_STAGE)
WHERE relative_path ILIKE '%.pdf'
"""
cur.execute(sql)
rows = cur.fetchall()
results = []
for row in rows:
    extracted = json.loads(row[1])
    extracted['file_name'] = row[0]
    results.append(extracted)
df = pd.DataFrame(results)[['file_name', 'responsible_person', 'deadline', 'risk_or_issue']]
display(Markdown("**異なるフォーマットに同一質問 → テンプレート不要！**"))
display(df)

---
## Part 4: AI_COMPLETE連携（要約パイプライン）
> AI_EXTRACTは構造化抽出特化。要約・翻訳などの自由記述にはAI_COMPLETEを組み合わせる

**AI_EXTRACT vs AI_COMPLETE 使い分け:**

| やりたいこと | 最適な関数 | 理由 |
|-------------|-----------|------|
| 定型フィールド抽出 | **AI_EXTRACT** | 1ステップ、JSON保証 |
| テーブル抽出 | **AI_EXTRACT** (schema) | 配列で返る |
| 全文テキスト化 | **AI_PARSE_DOCUMENT** | 2000頁対応 |
| 要約・翻訳 | AI_PARSE + **AI_COMPLETE** | 自由記述 |
| 大量ファイル一括 | **AI_EXTRACT** + DIRECTORY() | バッチ |

In [ ]:
sql = """
WITH parsed AS (
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@FF_STAGE', 'meeting_minutes.pdf'),
        {'mode': 'LAYOUT'}
    ):content::STRING AS doc_text
)
SELECT AI_COMPLETE(
    'mistral-large2',
    '以下の議事録の内容を3行で要約してください。主な決定事項と次のアクションを含めてください。\n\n' || doc_text
)::STRING AS summary
FROM parsed
"""
cur.execute(sql)
summary = cur.fetchone()[0]
print("=== AI_COMPLETE による議事録要約 ===\n")
print(summary)

---
## まとめ

### 従来AI OCR vs AI_EXTRACT

| 従来AI OCR | AI_EXTRACT |
|-----------|------------|
| テンプレート定義が必要（座標/アンカー） | 質問文を書くだけ |
| フォーマット変更→テンプレート作り直し | 同じ質問で対応 |
| 見たまま抽出のみ | 推論・判断・計算が可能 |
| フォーマットごとに個別対応 | 異なる文書に同一質問 |
| 導入に数週間～数ヶ月 | SQL 1行で即利用可能 |

### AI_COMPLETE vs 専用関数 使い分け（コスト込み）

| | AI_EXTRACT | AI_PARSE_DOCUMENT | AI_COMPLETE |
|---|---|---|---|
| 用途 | フィールド/テーブル抽出 | 全文テキスト化 | 要約/翻訳/判断 |
| 入力 | PDF直接(TO_FILE) | PDF直接(TO_FILE) | テキストのみ(事前パース必要) |
| 出力 | JSON保証 | Markdown/テキスト | 自由テキスト(JSON保証なし) |
| コスト | 5 cr/1Mトークン | OCR: 0.5 cr/1000頁 LAYOUT: 3.33 cr/1000頁 | モデル依存(mistral-large2: 2.4 cr/1Mトークン) |
| ステップ数 | 1ステップ | 1ステップ | 2ステップ(パース+推論) |
| バッチ | DIRECTORY()で容易 | DIRECTORY()で容易 | 事前テキスト化が必要 |
| 最適シーン | 定型抽出/帳票処理 | RAGチャンク化/アーカイブ | 要約/翻訳/自由質問 |

**結論:** フィールド抽出ならAI_EXTRACTが圧倒的に効率的（1ステップ/JSON保証/低コスト）。自由な分析が必要な場合のみAI_COMPLETEを組み合わせる。

### キーメッセージ
1. **テンプレート不要** — 質問を書くだけ
2. **推論ができる** — 出荷可能か等の判断抽出
3. **レイアウト非依存** — 異なるフォーマットに同一質問
4. **表記ゆれに強い** — 合計/TOTAL/お支払い金額 → 全て同じフィールド
5. **即時利用可能** — セットアップ数分

### Next Step
- Task + DIRECTORY() で新規ファイル到着時に自動抽出
- Cortex Search と組み合わせて RAG パイプライン構築
- AI_CLASSIFY でドキュメント自動分類 → 種類別に抽出ロジックを分岐